# LSTM Forex Trading Model
### Recreating: Predictive modeling of foreign exchange trading signals using machine learning techniques

**Authors:** Sugarbayar Enkhbayar, Robert Ślepaczuk  
**Journal:** Expert Systems With Applications (2025)

---

This notebook continues from the feature engineering phase and implements:
- **Step 3:** Target variable preparation
- **Step 4:** Walk-forward optimization
- **Step 5:** LSTM model training
- **Step 6:** Signal generation
- **Step 7:** Backtesting with transaction costs
- **Step 8:** Performance evaluation

In [2]:
# Configure Keras to use PyTorch backend
import os
os.environ['KERAS_BACKEND'] = 'torch'

# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ML & Deep Learning
import keras
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout
from keras.optimizers import Adam
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
plt.style.use('seaborn-v0_8-darkgrid')

print(f"Keras version: {keras.__version__}")
print(f"Backend: {keras.backend.backend()}")
print("Libraries imported successfully!")

Keras version: 3.12.0
Backend: torch
Libraries imported successfully!


In [ ]:
# Load the data with features from Step 2
df = pd.read_csv('data/EURUSD_1day_with_features.csv', index_col='date', parse_dates=True)

print(f"Data loaded successfully!")
print(f"Shape: {df.shape}")
print(f"Date range: {df.index.min()} to {df.index.max()}")
print(f"\nColumns: {list(df.columns)}")

---

## Step 3: Target Variable Preparation

**Status**: Complete

Target variables (simple_return and log_return) were already calculated in Step 1:
- **Simple return**: `(Close_t - Close_{t-1}) / Close_{t-1}`
- **Log return**: `log(Close_t) - log(Close_{t-1})`

Both are available in `EURUSD_1day_with_features.csv` and will be used as prediction targets.

For this implementation:
- **Focus**: LSTM model only
- **Data**: EURUSD daily frequency
- **Target**: Simple return (can be changed to log return)

In [4]:
# Walk-forward optimization parameters
TRAIN_DAYS = 600   # 68% - Training set
VAL_DAYS = 156     # 18% - Validation set (for hyperparameter tuning)
TEST_DAYS = 126    # 14% - Test set (for predictions)
ROLL_DAYS = 126    # 6 months rolling window (approximately)

WINDOW_SIZE = TRAIN_DAYS + VAL_DAYS + TEST_DAYS  # 882 days total per window

# LSTM-specific parameters
SEQUENCE_LENGTH = 20  # Lookback period for LSTM sequences
TARGET_COLUMN = 'simple_return'  # Can change to 'log_return'

# Feature columns (all technical indicators)
FEATURE_COLS = [
    # Statistical indicators
    'momentum', 'avg_price', 'range', 'ohlc',
    # EMAs
    'ema_10', 'ema_20', 'ema_50', 'ema_100', 'ema_200',
    # MACD
    'macd', 'macd_signal', 'macd_hist',
    # ADX
    'adx', 'plus_di', 'minus_di',
    # Oscillators
    'rsi', 'stoch_k', 'stoch_d', 'cci', 'williams_r',
    # Bollinger Bands
    'bb_upper', 'bb_middle', 'bb_lower', 'bb_width', 'bb_position',
    # ATR
    'atr'
]

print(f"Walk-forward configuration:")
print(f"  Train: {TRAIN_DAYS} days")
print(f"  Validation: {VAL_DAYS} days")
print(f"  Test: {TEST_DAYS} days")
print(f"  Rolling: {ROLL_DAYS} days")
print(f"  Sequence length: {SEQUENCE_LENGTH} days")
print(f"  Total features: {len(FEATURE_COLS)}")
print(f"  Target: {TARGET_COLUMN}")

Walk-forward configuration:
  Train: 600 days
  Validation: 156 days
  Test: 126 days
  Rolling: 126 days
  Sequence length: 20 days
  Total features: 26
  Target: simple_return


In [5]:
# Drop rows with NaN values (from technical indicators)
df_clean = df.dropna()

print(f"Original data shape: {df.shape}")
print(f"After dropping NaNs: {df_clean.shape}")
print(f"Rows dropped: {df.shape[0] - df_clean.shape[0]}")
print(f"\nDate range: {df_clean.index.min()} to {df_clean.index.max()}")
print(f"\nFirst few rows:")
df_clean[FEATURE_COLS + [TARGET_COLUMN]].head()

Original data shape: (6743, 33)
After dropping NaNs: (6705, 33)
Rows dropped: 38

Date range: 2000-02-24 00:00:00 to 2025-11-07 00:00:00

First few rows:


,momentum,avg_price,range,ohlc,ema_10,ema_20,ema_50,ema_100,ema_200,macd,macd_signal,macd_hist,adx,plus_di,minus_di,rsi,stoch_k,stoch_d,cci,williams_r,bb_upper,bb_middle,bb_lower,bb_width,bb_position,atr,simple_return
date,,,,,,,,,,,,,,,,,,,,,,,,,,,
2000-02-24,0.0121,0.99656,0.0153,0.99706,0.991906,0.992246,1.001580,1.010179,1.016453,-0.002595,-0.005556,0.002961,31.783824,29.754204,14.424321,55.469755,47.462687,73.401961,39.986872,-52.537313,1.003041,0.985560,0.968079,0.034963,0.670181,0.011043,-0.012056
2000-02-25,0.0174,0.98261,0.0200,0.98271,0.988670,0.990519,1.000503,1.009465,1.016032,-0.003787,-0.005202,0.001415,30.286295,27.577938,23.141487,45.997866,4.109589,45.131935,-27.622323,-95.890411,1.003258,0.985395,0.967532,0.035727,0.184130,0.011914,-0.017549
2000-02-28,0.0032,0.95691,0.0356,0.96471,0.985441,0.988651,0.999342,1.008701,1.015583,-0.004934,-0.005149,0.000215,31.313127,19.957761,38.067582,42.274678,45.428571,32.333616,-118.584056,-54.571429,1.003294,0.985380,0.967466,0.035828,0.096125,0.013529,-0.003285
2000-02-29,0.0073,0.97646,0.0273,0.97186,0.981472,0.986266,0.997941,1.007808,1.015066,-0.006358,-0.005391,-0.000968,31.889314,23.414634,35.170732,33.077766,35.000000,28.179387,-71.937016,-65.000000,1.004457,0.984990,0.965523,0.038934,-0.049139,0.014643,-0.007519
2000-03-01,-0.0094,0.96991,0.0140,0.96911,0.979933,0.985004,0.996963,1.007119,1.014647,-0.006652,-0.005643,-0.001009,32.678379,23.300971,34.126214,43.278689,48.428571,42.952381,-84.144568,-51.571429,1.004680,0.984790,0.964900,0.039779,0.203866,0.014714,0.009755


In [6]:
def generate_windows(df, window_size, roll_days, min_windows=40):
    """
    Generate rolling walk-forward windows.
    
    Parameters:
    - df: DataFrame with features and target
    - window_size: Total size of one window (train + val + test)
    - roll_days: Number of days to roll forward
    - min_windows: Minimum number of windows to generate
    
    Returns:
    - List of window dictionaries with start/end indices for train/val/test
    """
    windows = []
    start_idx = 0
    
    while start_idx + window_size <= len(df):
        end_idx = start_idx + window_size
        
        # Split into train, val, test
        train_end = start_idx + TRAIN_DAYS
        val_end = train_end + VAL_DAYS
        test_end = val_end + TEST_DAYS
        
        window = {
            'window_id': len(windows),
            'train_start': start_idx,
            'train_end': train_end,
            'val_start': train_end,
            'val_end': val_end,
            'test_start': val_end,
            'test_end': test_end,
            'date_start': df.index[start_idx],
            'date_end': df.index[test_end - 1]
        }
        
        windows.append(window)
        
        # Roll forward
        start_idx += roll_days
        
        # Stop if we have enough windows
        if len(windows) >= min_windows:
            break
    
    return windows

# Generate windows
windows = generate_windows(df_clean, WINDOW_SIZE, ROLL_DAYS, min_windows=40)

print(f"Generated {len(windows)} windows")
print(f"\nFirst window:")
print(f"  Train: {windows[0]['date_start']} to {df_clean.index[windows[0]['train_end']-1]}")
print(f"  Val:   {df_clean.index[windows[0]['val_start']]} to {df_clean.index[windows[0]['val_end']-1]}")
print(f"  Test:  {df_clean.index[windows[0]['test_start']]} to {windows[0]['date_end']}")
print(f"\nLast window:")
print(f"  Train: {windows[-1]['date_start']} to {df_clean.index[windows[-1]['train_end']-1]}")
print(f"  Val:   {df_clean.index[windows[-1]['val_start']]} to {df_clean.index[windows[-1]['val_end']-1]}")
print(f"  Test:  {df_clean.index[windows[-1]['test_start']]} to {windows[-1]['date_end']}")

Generated 40 windows

First window:
  Train: 2000-02-24 00:00:00 to 2002-06-14 00:00:00
  Val:   2002-06-17 00:00:00 to 2003-01-22 00:00:00
  Test:  2003-01-23 00:00:00 to 2003-07-17 00:00:00

Last window:
  Train: 2019-02-07 00:00:00 to 2021-06-01 00:00:00
  Val:   2021-06-02 00:00:00 to 2022-01-05 00:00:00
  Test:  2022-01-06 00:00:00 to 2022-06-30 00:00:00


In [7]:
from sklearn.preprocessing import MinMaxScaler

def create_sequences(data, target, sequence_length):
    """
    Create sequences for LSTM input.
    
    Parameters:
    - data: Feature array (samples, features)
    - target: Target array (samples,)
    - sequence_length: Number of time steps to look back
    
    Returns:
    - X_sequences: (samples - sequence_length, sequence_length, features)
    - y_sequences: (samples - sequence_length,)
    """
    X_sequences = []
    y_sequences = []
    
    for i in range(sequence_length, len(data)):
        X_sequences.append(data[i - sequence_length:i])
        y_sequences.append(target[i])
    
    return np.array(X_sequences), np.array(y_sequences)

def prepare_window_data(df, window, feature_cols, target_col, sequence_length):
    """
    Prepare train, validation, and test data for one window.
    
    Returns:
    - Dictionary with scaled sequences for train, val, test
    - Scaler fitted on training data
    """
    # Extract indices
    train_data = df.iloc[window['train_start']:window['train_end']]
    val_data = df.iloc[window['val_start']:window['val_end']]
    test_data = df.iloc[window['test_start']:window['test_end']]
    
    # Fit scaler on training data only
    scaler = MinMaxScaler()
    train_features_scaled = scaler.fit_transform(train_data[feature_cols])
    val_features_scaled = scaler.transform(val_data[feature_cols])
    test_features_scaled = scaler.transform(test_data[feature_cols])
    
    # Get targets
    train_target = train_data[target_col].values
    val_target = val_data[target_col].values
    test_target = test_data[target_col].values
    
    # Create sequences
    X_train, y_train = create_sequences(train_features_scaled, train_target, sequence_length)
    X_val, y_val = create_sequences(val_features_scaled, val_target, sequence_length)
    X_test, y_test = create_sequences(test_features_scaled, test_target, sequence_length)
    
    return {
        'X_train': X_train, 'y_train': y_train,
        'X_val': X_val, 'y_val': y_val,
        'X_test': X_test, 'y_test': y_test,
        'scaler': scaler
    }

# Test on first window
test_data = prepare_window_data(df_clean, windows[0], FEATURE_COLS, TARGET_COLUMN, SEQUENCE_LENGTH)

print(f"Window 0 data shapes:")
print(f"  X_train: {test_data['X_train'].shape} (samples, sequence_length, features)")
print(f"  y_train: {test_data['y_train'].shape}")
print(f"  X_val: {test_data['X_val'].shape}")
print(f"  y_val: {test_data['y_val'].shape}")
print(f"  X_test: {test_data['X_test'].shape}")
print(f"  y_test: {test_data['y_test'].shape}")

Window 0 data shapes:
  X_train: (580, 20, 26) (samples, sequence_length, features)
  y_train: (580,)
  X_val: (136, 20, 26)
  y_val: (136,)
  X_test: (106, 20, 26)
  y_test: (106,)


In [8]:
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout
from keras.optimizers import SGD, RMSprop
from keras.callbacks import EarlyStopping

def build_lstm_model(input_shape, n_layers, n_neurons, learning_rate, momentum, optimizer_name):
    """
    Build LSTM model according to hyperparameters.
    
    Parameters:
    - input_shape: (sequence_length, n_features)
    - n_layers: Number of LSTM layers (1-3)
    - n_neurons: Number of neurons per layer (5-40)
    - learning_rate: Learning rate (0.001-0.05)
    - momentum: Momentum for SGD (0.1-0.4)
    - optimizer_name: 'SGD' or 'RMSprop'
    
    Returns:
    - Compiled Keras model
    """
    model = Sequential()
    
    # First LSTM layer
    if n_layers == 1:
        model.add(LSTM(n_neurons, activation='tanh', input_shape=input_shape))
    else:
        model.add(LSTM(n_neurons, activation='tanh', return_sequences=True, input_shape=input_shape))
    model.add(Dropout(0.2))  # Fixed dropout rate from paper
    
    # Additional LSTM layers
    for i in range(1, n_layers):
        if i == n_layers - 1:  # Last LSTM layer
            model.add(LSTM(n_neurons, activation='tanh'))
        else:
            model.add(LSTM(n_neurons, activation='tanh', return_sequences=True))
        model.add(Dropout(0.2))
    
    # Output layer (regression)
    model.add(Dense(1))
    
    # Choose optimizer
    if optimizer_name == 'SGD':
        optimizer = SGD(learning_rate=learning_rate, momentum=momentum)
    else:  # RMSprop
        optimizer = RMSprop(learning_rate=learning_rate, momentum=momentum)
    
    # Compile with MAE loss (used in paper)
    model.compile(optimizer=optimizer, loss='mae', metrics=['mae'])
    
    return model

# Test model building
test_model = build_lstm_model(
    input_shape=(SEQUENCE_LENGTH, len(FEATURE_COLS)),
    n_layers=2,
    n_neurons=20,
    learning_rate=0.01,
    momentum=0.3,
    optimizer_name='SGD'
)

print("LSTM Model Architecture:")
test_model.summary()

LSTM Model Architecture:


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                          │ (None, 20, 20)              │           3,760 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 20, 20)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ (None, 20)                  │           3,280 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 20)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 1)                   │              21 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 7,061 (27.58 KB)

 Trainable params: 7,061 (27.58 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
from scipy.stats import randint, uniform, loguniform
import time

# Define hyperparameter space from Table 4
param_space = {
    'n_layers': [1, 2, 3],
    'n_neurons': list(range(5, 41)),  # 5-40
    'optimizer': ['SGD', 'RMSprop'],
    'batch_size': [32, 64, 128],
    'epochs': [10, 20, 30]
}

def sample_hyperparameters():
    """Sample hyperparameters from the defined space."""
    import random
    return {
        'n_layers': random.choice(param_space['n_layers']),
        'n_neurons': random.choice(param_space['n_neurons']),
        'learning_rate': 10 ** np.random.uniform(-3, -1.3),  # loguniform(0.001, 0.05)
        'momentum': np.random.uniform(0.1, 0.4),
        'optimizer': random.choice(param_space['optimizer']),
        'batch_size': random.choice(param_space['batch_size']),
        'epochs': random.choice(param_space['epochs'])
    }

def train_and_evaluate(X_train, y_train, X_val, y_val, params, verbose=0):
    """
    Train model with given hyperparameters and return validation MAE.
    """
    model = build_lstm_model(
        input_shape=(X_train.shape[1], X_train.shape[2]),
        n_layers=params['n_layers'],
        n_neurons=params['n_neurons'],
        learning_rate=params['learning_rate'],
        momentum=params['momentum'],
        optimizer_name=params['optimizer']
    )
    
    # Early stopping to prevent overfitting
    early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    
    # Train
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=params['epochs'],
        batch_size=params['batch_size'],
        callbacks=[early_stop],
        verbose=verbose
    )
    
    # Get best validation MAE
    val_mae = min(history.history['val_mae'])
    
    return val_mae, model

def randomized_search(X_train, y_train, X_val, y_val, n_iter=20):
    """
    Randomized search over hyperparameters (20 iterations as per paper).
    """
    best_mae = float('inf')
    best_params = None
    best_model = None
    
    print(f"Starting randomized search with {n_iter} iterations...")
    
    for i in range(n_iter):
        # Sample hyperparameters
        params = sample_hyperparameters()
        
        print(f"\nIteration {i+1}/{n_iter}")
        print(f"  Params: layers={params['n_layers']}, neurons={params['n_neurons']}, "
              f"lr={params['learning_rate']:.4f}, opt={params['optimizer']}, "
              f"batch={params['batch_size']}, epochs={params['epochs']}")
        
        # Train and evaluate
        try:
            val_mae, model = train_and_evaluate(X_train, y_train, X_val, y_val, params, verbose=0)
            
            print(f"  Val MAE: {val_mae:.6f}")
            
            # Update best
            if val_mae < best_mae:
                best_mae = val_mae
                best_params = params
                best_model = model
                print(f"  --> New best!")
        
        except Exception as e:
            print(f"  Error: {str(e)}")
            continue
    
    print(f"\n=== Search Complete ===")
    print(f"Best MAE: {best_mae:.6f}")
    print(f"Best params: {best_params}")
    
    return best_model, best_params, best_mae

print("Hyperparameter search function ready")
print(f"Search space size: ~{len(param_space['n_layers']) * len(param_space['n_neurons']) * len(param_space['optimizer']) * len(param_space['batch_size']) * len(param_space['epochs'])} combinations")

Hyperparameter search function ready
Search space size: ~1944 combinations


In [ ]:
import os
import pickle

# Load existing results if they exist (for resuming)
results_file = 'lstm_results_checkpoint.pkl'
if os.path.exists(results_file):
    with open(results_file, 'rb') as f:
        all_results = pickle.load(f)
    print(f"Loaded {len(all_results)} existing results. Resuming from window {len(all_results) + 1}...")
else:
    all_results = []
    print("Starting fresh training...")

# NOTE: Training 40 windows will take a long time!
# For testing, you can change n_windows to a smaller number (e.g., 3)
n_windows = len(windows)  # Full run: 40 windows
# n_windows = 3  # Quick test: 3 windows

start_window = len(all_results)  # Resume from where we left off

print(f"Training LSTM on windows {start_window + 1} to {n_windows}...")
print(f"Estimated time: ~{(n_windows - start_window) * 20 * 2} minutes (assuming 2 min per iteration)")
print("="*60)

for window_idx in range(start_window, n_windows):
    window = windows[window_idx]
    print(f"\n{'='*60}")
    print(f"WINDOW {window_idx + 1}/{n_windows}")
    print(f"Date range: {window['date_start']} to {window['date_end']}")
    print(f"{'='*60}")
    
    start_time = time.time()
    
    # Prepare data for this window
    window_data = prepare_window_data(
        df_clean, window, FEATURE_COLS, TARGET_COLUMN, SEQUENCE_LENGTH
    )
    
    X_train = window_data['X_train']
    y_train = window_data['y_train']
    X_val = window_data['X_val']
    y_val = window_data['y_val']
    X_test = window_data['X_test']
    y_test = window_data['y_test']
    
    print(f"\nData shapes:")
    print(f"  Train: {X_train.shape}")
    print(f"  Val:   {X_val.shape}")
    print(f"  Test:  {X_test.shape}")
    
    # Hyperparameter search on train + validation
    print(f"\nStarting hyperparameter search (20 iterations)...")
    best_model, best_params, best_mae = randomized_search(
        X_train, y_train, X_val, y_val, n_iter=20
    )
    
    # Make predictions on test set
    y_pred = best_model.predict(X_test, verbose=0).flatten()
    
    # Calculate test MAE
    test_mae = np.mean(np.abs(y_test - y_pred))
    
    # Store results
    result = {
        'window_id': window_idx,
        'date_start': str(window['date_start']),
        'date_end': str(window['date_end']),
        'best_params': best_params,
        'val_mae': best_mae,
        'test_mae': test_mae,
        'predictions': y_pred.tolist(),
        'actuals': y_test.tolist()
    }
    all_results.append(result)
    
    elapsed = time.time() - start_time
    print(f"\nWindow {window_idx + 1} complete in {elapsed/60:.1f} minutes")
    print(f"  Val MAE:  {best_mae:.6f}")
    print(f"  Test MAE: {test_mae:.6f}")
    
    # SAVE CHECKPOINT AFTER EACH WINDOW
    with open(results_file, 'wb') as f:
        pickle.dump(all_results, f)
    print(f"  Checkpoint saved: {len(all_results)} windows complete")

print(f"\n{'='*60}")
print(f"ALL WINDOWS COMPLETE")
print(f"{'='*60}")

# Save final results
with open('lstm_results.pkl', 'wb') as f:
    pickle.dump(all_results, f)
print("\nFinal results saved to: lstm_results.pkl")

In [13]:
import pickle
import json

# Save results to pickle file
with open('lstm_results.pkl', 'wb') as f:
    pickle.dump(all_results, f)

# Also save to JSON for readability (without predictions/actuals to keep file small)
results_summary = []
for r in all_results:
    summary = {k: v for k, v in r.items() if k not in ['predictions', 'actuals']}
    results_summary.append(summary)

with open('lstm_results_summary.json', 'w') as f:
    json.dump(results_summary, f, indent=2)

print("Results saved:")
print("  lstm_results.pkl (full results with predictions)")
print("  lstm_results_summary.json (summary without predictions)")

# Display average performance
avg_val_mae = np.mean([r['val_mae'] for r in all_results])
avg_test_mae = np.mean([r['test_mae'] for r in all_results])

print(f"\nAverage Performance Across All Windows:")
print(f"  Validation MAE: {avg_val_mae:.6f}")
print(f"  Test MAE:       {avg_test_mae:.6f}")

Results saved:
  lstm_results.pkl (full results with predictions)
  lstm_results_summary.json (summary without predictions)

Average Performance Across All Windows:
  Validation MAE: 0.004896
  Test MAE:       0.005474


---

## Step 6: Signal Generation

Generate trading signals from LSTM predictions using quartile thresholds:
- **Buy signal (1)**: Predicted return > Q3 (upper quartile)
- **Sell signal (-1)**: Predicted return < Q1 (lower quartile)
- **No position (0)**: Q1 <= Predicted return <= Q3

Three signal strategies:
1. **Buy & Sell**: Long when Buy, Short when Sell, otherwise no position
2. **Only Buy**: Long when Buy, otherwise no position
3. **Only Sell**: Short when Sell, otherwise no position

In [14]:
import pickle
import json

# Load results
with open('lstm_results.pkl', 'rb') as f:
    all_results = pickle.load(f)

print(f"Loaded results for {len(all_results)} windows")
print(f"\nFirst window info:")
print(f"  Date range: {all_results[0]['date_start']} to {all_results[0]['date_end']}")
print(f"  Val MAE: {all_results[0]['val_mae']:.6f}")
print(f"  Test MAE: {all_results[0]['test_mae']:.6f}")
print(f"  Number of predictions: {len(all_results[0]['predictions'])}")

Loaded results for 3 windows

First window info:
  Date range: 2000-02-24 00:00:00 to 2003-07-17 00:00:00
  Val MAE: 0.004498
  Test MAE: 0.005361
  Number of predictions: 106


In [ ]:
def generate_signals_quartile(predictions, strategy='buy_sell'):
    """
    Generate trading signals based on strategy from paper.
    
    Parameters:
    - predictions: Array of predicted returns
    - strategy: 'buy_sell', 'only_buy', or 'only_sell'
    
    Returns:
    - signals: Array of trading signals (1=long, -1=short, 0=no position)
    """
    predictions = np.array(predictions)
    
    # Initialize signals
    signals = np.zeros(len(predictions))
    
    if strategy == 'buy_sell':
        # Buy when predicted return > Q3, Sell when < Q1 (Equation 33)
        q1 = np.percentile(predictions, 25)
        q3 = np.percentile(predictions, 75)
        signals[predictions >= q3] = 1
        signals[predictions <= q1] = -1
        return signals, q1, q3
        
    elif strategy == 'only_buy':
        # Buy when predicted return > 0 (Equation 34)
        signals[predictions > 0] = 1
        return signals, None, 0  # threshold is 0
        
    elif strategy == 'only_sell':
        # Sell when predicted return < 0 (Equation 35)
        signals[predictions < 0] = -1
        return signals, 0, None  # threshold is 0

def add_signals_to_results(all_results, strategy='buy_sell'):
    """
    Add trading signals to results for all windows.
    
    Parameters:
    - all_results: List of result dictionaries
    - strategy: Signal generation strategy
    
    Returns:
    - Updated results with signals
    """
    for result in all_results:
        signals, threshold_low, threshold_high = generate_signals_quartile(
            result['predictions'], strategy
        )
        result['signals'] = signals.tolist()
        result['signal_strategy'] = strategy
        
        # Store thresholds
        if strategy == 'buy_sell':
            result['threshold_q1'] = threshold_low
            result['threshold_q3'] = threshold_high
        elif strategy == 'only_buy':
            result['threshold'] = threshold_high  # 0
        elif strategy == 'only_sell':
            result['threshold'] = threshold_low  # 0
        
        # Calculate signal distribution
        n_buy = np.sum(signals == 1)
        n_sell = np.sum(signals == -1)
        n_hold = np.sum(signals == 0)
        
        result['signal_distribution'] = {
            'buy': int(n_buy),
            'sell': int(n_sell),
            'hold': int(n_hold)
        }
    
    return all_results

print("Signal generation functions ready (FIXED - using 0 threshold for only_buy/only_sell)")

In [ ]:
# Generate signals for all three strategies
strategies = ['buy_sell', 'only_buy', 'only_sell']
results_by_strategy = {}

for strategy in strategies:
    print(f"\n{'='*60}")
    print(f"Strategy: {strategy.upper()}")
    print(f"{'='*60}")
    
    # Deep copy results to avoid modifying original
    import copy
    strategy_results = copy.deepcopy(all_results)
    
    # Add signals
    strategy_results = add_signals_to_results(strategy_results, strategy)
    results_by_strategy[strategy] = strategy_results
    
    # Display signal statistics
    total_buy = sum(r['signal_distribution']['buy'] for r in strategy_results)
    total_sell = sum(r['signal_distribution']['sell'] for r in strategy_results)
    total_hold = sum(r['signal_distribution']['hold'] for r in strategy_results)
    total_signals = total_buy + total_sell + total_hold
    
    print(f"\nSignal Distribution (across all windows):")
    print(f"  Buy:  {total_buy:4d} ({total_buy/total_signals*100:5.1f}%)")
    print(f"  Sell: {total_sell:4d} ({total_sell/total_signals*100:5.1f}%)")
    print(f"  Hold: {total_hold:4d} ({total_hold/total_signals*100:5.1f}%)")
    print(f"  Total: {total_signals}")
    
    # Show first window as example
    print(f"\nWindow 1 example:")
    if strategy == 'buy_sell':
        print(f"  Thresholds: Q1={strategy_results[0]['threshold_q1']:.6f}, Q3={strategy_results[0]['threshold_q3']:.6f}")
    else:
        print(f"  Threshold: {strategy_results[0]['threshold']:.6f} (0 = any positive/negative prediction)")
    print(f"  Signals: Buy={strategy_results[0]['signal_distribution']['buy']}, "
          f"Sell={strategy_results[0]['signal_distribution']['sell']}, "
          f"Hold={strategy_results[0]['signal_distribution']['hold']}")

print(f"\n{'='*60}")
print("Signal generation complete for all strategies")
print(f"{'='*60}")

In [ ]:
# Save results with signals for each strategy
for strategy, results in results_by_strategy.items():
    # Save full results with signals
    with open(f'lstm_results_{strategy}.pkl', 'wb') as f:
        pickle.dump(results, f)
    
    # Save summary (handle different threshold formats)
    summary = []
    for r in results:
        s = {
            'window_id': r['window_id'],
            'date_start': r['date_start'],
            'date_end': r['date_end'],
            'val_mae': r['val_mae'],
            'test_mae': r['test_mae'],
            'signal_strategy': r['signal_strategy'],
            'signal_distribution': r['signal_distribution']
        }
        
        # Add thresholds based on strategy
        if strategy == 'buy_sell':
            s['threshold_q1'] = r['threshold_q1']
            s['threshold_q3'] = r['threshold_q3']
        else:
            s['threshold'] = r['threshold']
        
        summary.append(s)
    
    with open(f'lstm_results_{strategy}_summary.json', 'w') as f:
        json.dump(summary, f, indent=2)

print("Results with signals saved for all strategies:")
for strategy in strategies:
    print(f"  lstm_results_{strategy}.pkl")
    print(f"  lstm_results_{strategy}_summary.json")

---

## Step 7: Backtesting with Transaction Costs

Backtest each strategy with realistic assumptions:
- **Transaction cost**: 0.02% (0.0002) per trade
- **Initial capital**: $10,000
- **Position sizing**: Full capital allocation
- **Slippage**: Included in transaction cost

Calculate returns for each trade and accumulate to get equity curve.

In [ ]:
def backtest_strategy_forex_dynamic(actuals, signals, df_prices, test_indices, initial_capital, transaction_cost_pct=0.0002):
    """
    Backtest a trading strategy using proper forex mechanics with DYNAMIC position sizing.
    
    Forex parameters (from paper):
    - Initial capital: $1000 (or carried forward from previous window)
    - Lot size: DYNAMIC - scales with capital (capital/1000 micro lots)
    - Pip value: $0.1 per pip
    - Pip = 0.0001 for EURUSD
    
    Parameters:
    - actuals: Not used (we calculate from prices directly)
    - signals: Array of trading signals (1=long, -1=short, 0=no position)
    - df_prices: DataFrame with Open/Close prices
    - test_indices: Indices in df_prices corresponding to test period
    - initial_capital: Starting capital for this window
    - transaction_cost_pct: Transaction cost (0.02% = 0.0002)
    
    Returns:
    - Dictionary with backtest results
    """
    # Constants
    PIP_VALUE = 0.1  # $0.1 per pip
    PIP_SIZE = 0.0001  # 1 pip for EURUSD
    
    capital = initial_capital
    position = 0  # 0=no position, 1=long, -1=short
    entry_price = 0.0
    entry_lot_size = 0.0
    
    equity_curve = [capital]
    returns = []
    trades = []
    
    # Get Open and Close prices for test period
    test_data = df_prices.iloc[test_indices]
    opens = test_data['Open'].values
    closes = test_data['Close'].values
    
    for i in range(len(signals)):
        signal = signals[i]
        open_price = opens[i]
        close_price = closes[i]
        
        # Check if we need to exit or enter
        if position != 0 and signal != position:
            # Exit current position at open of this bar
            exit_price = open_price
            
            # Calculate P&L in pips
            if position == 1:  # Long
                pips = (exit_price - entry_price) / PIP_SIZE
            else:  # Short
                pips = (entry_price - exit_price) / PIP_SIZE
            
            # Calculate P&L in dollars using LOT SIZE AT ENTRY
            pnl = pips * PIP_VALUE * entry_lot_size / 1000
            
            # Apply transaction cost (exit)
            cost = capital * transaction_cost_pct
            pnl -= cost
            
            # Update capital
            capital += pnl
            
            # Record trade
            trade_return = pnl / (capital - pnl)  # Return on capital before trade
            returns.append(trade_return)
            trades.append({
                'type': 'long' if position == 1 else 'short',
                'entry': entry_price,
                'exit': exit_price,
                'pips': pips,
                'lot_size': entry_lot_size,
                'pnl': pnl,
                'return': trade_return
            })
            
            position = 0
        
        # Enter new position if signal says so
        if position == 0 and signal != 0:
            # Pay transaction cost (entry)
            cost = capital * transaction_cost_pct
            capital -= cost
            
            # Enter at open of this bar with DYNAMIC lot sizing
            entry_price = open_price
            position = signal
            entry_lot_size = capital  # DYNAMIC: lot size scales with capital
        
        equity_curve.append(capital)
    
    # Close any open position at end
    if position != 0:
        exit_price = closes[-1]
        
        if position == 1:
            pips = (exit_price - entry_price) / PIP_SIZE
        else:
            pips = (entry_price - exit_price) / PIP_SIZE
        
        pnl = pips * PIP_VALUE * entry_lot_size / 1000
        cost = capital * transaction_cost_pct
        pnl -= cost
        capital += pnl
        
        trade_return = pnl / (capital - pnl)
        returns.append(trade_return)
        trades.append({
            'type': 'long' if position == 1 else 'short',
            'entry': entry_price,
            'exit': exit_price,
            'pips': pips,
            'lot_size': entry_lot_size,
            'pnl': pnl,
            'return': trade_return
        })
    
    # Calculate metrics
    final_return = (capital - initial_capital) / initial_capital
    
    winning_trades = [t for t in trades if t['pnl'] > 0]
    losing_trades = [t for t in trades if t['pnl'] <= 0]
    
    return {
        'equity_curve': equity_curve,
        'trades': trades,
        'strategy_returns': returns,
        'final_capital': capital,
        'final_return': final_return,
        'n_trades': len(trades),
        'n_long': len([t for t in trades if t['type'] == 'long']),
        'n_short': len([t for t in trades if t['type'] == 'short']),
        'n_winning': len(winning_trades),
        'n_losing': len(losing_trades)
    }

def backtest_all_windows_forex_dynamic(results_by_strategy, df_clean, windows, initial_capital=1000.0, transaction_cost=0.0002):
    """
    Backtest all windows for all strategies using forex mechanics with dynamic position sizing.
    Capital is carried forward across windows to allow compounding.
    """
    for strategy, results in results_by_strategy.items():
        capital = initial_capital  # Start with $1000
        
        for window_idx, result in enumerate(results):
            window = windows[window_idx]
            
            # Get test indices
            test_start = window['test_start']
            test_end = window['test_end']
            test_indices = range(test_start, test_end)
            
            # Run backtest starting with current capital
            backtest = backtest_strategy_forex_dynamic(
                result['actuals'],
                result['signals'],
                df_clean,
                test_indices,
                capital,  # Use capital from previous window
                transaction_cost
            )
            result['backtest'] = backtest
            
            # Carry capital forward to next window
            capital = backtest['final_capital']
    
    return results_by_strategy

print(\"Forex backtesting functions ready (DYNAMIC position sizing + capital compounding)\")
print(\"  - Lot size scales with capital: lot_size = capital (in units)\")
print(\"  - At $1000: trade 1000 units (1 micro lot)\")
print(\"  - At $2000: trade 2000 units (2 micro lots)\")
print(\"  - Capital carries forward across windows for compounding\")


In [ ]:
# Run backtests for all strategies using FOREX mechanics with DYNAMIC position sizing
TRANSACTION_COST = 0.0002  # 0.02% per trade (as per paper)
INITIAL_CAPITAL = 1000.0   # $1000 starting capital

print(\"Running forex backtests with DYNAMIC position sizing...\")
print(f\"Transaction cost: {TRANSACTION_COST*100:.2f}% per trade\")
print(f\"Initial capital: ${INITIAL_CAPITAL:.0f}\")
print(f\"Position sizing: DYNAMIC (lot size = capital/1000 micro lots)\")
print(f\"Capital compounding: YES (carries forward across windows)\\n\")

results_by_strategy = backtest_all_windows_forex_dynamic(
    results_by_strategy, 
    df_clean, 
    windows,
    INITIAL_CAPITAL,
    TRANSACTION_COST
)

# Display results for each strategy
for strategy in strategies:
    results = results_by_strategy[strategy]
    
    print(f\"{'='*60}\")
    print(f\"Strategy: {strategy.upper()}\")
    print(f\"{'='*60}\")
    
    # Final capital after all windows
    final_capital = results[-1]['backtest']['final_capital']
    total_return = (final_capital - INITIAL_CAPITAL) / INITIAL_CAPITAL
    total_trades = sum(r['backtest']['n_trades'] for r in results)
    total_long = sum(r['backtest']['n_long'] for r in results)
    total_short = sum(r['backtest']['n_short'] for r in results)
    
    print(f\"\\nFinal Results (after {len(results)} windows):\")
    print(f\"  Initial Capital: ${INITIAL_CAPITAL:.2f}\")
    print(f\"  Final Capital:   ${final_capital:.2f}\")
    print(f\"  Total Return:    {total_return*100:.2f}%\")
    print(f\"  Total Trades:    {total_trades}\")
    print(f\"    Long:  {total_long} ({total_long/total_trades*100:.1f}%)\")
    print(f\"    Short: {total_short} ({total_short/total_trades*100:.1f}%)\")
    
    # Show capital progression
    print(f\"\\nCapital Progression:\")
    print(f\"  Window 1:  ${results[0]['backtest']['final_capital']:.2f}\")
    print(f\"  Window 10: ${results[9]['backtest']['final_capital']:.2f}\")
    print(f\"  Window 20: ${results[19]['backtest']['final_capital']:.2f}\")
    print(f\"  Window 30: ${results[29]['backtest']['final_capital']:.2f}\")
    print(f\"  Window 40: ${results[39]['backtest']['final_capital']:.2f}\")

print(f\"\\n{'='*60}\")
print(\"Forex backtesting complete (DYNAMIC position sizing)\")
print(f\"{'='*60}\")


---

## Step 8: Performance Evaluation

Calculate key performance metrics:
- **Total Return**: Cumulative return over all windows
- **Annualized Return**: Return annualized (252 trading days/year)
- **Sharpe Ratio**: Risk-adjusted return (excess return / volatility)
- **Maximum Drawdown**: Largest peak-to-trough decline
- **Win Rate**: Percentage of profitable trades
- **Average Win/Loss**: Mean return on winning vs losing trades

In [20]:
def calculate_performance_metrics(results, risk_free_rate=0.0):
    """
    Calculate comprehensive performance metrics for a strategy.
    
    Parameters:
    - results: List of window results with backtest data
    - risk_free_rate: Annual risk-free rate (default 0%)
    
    Returns:
    - Dictionary of performance metrics
    """
    # Concatenate all strategy returns
    all_returns = []
    for r in results:
        all_returns.extend(r['backtest']['strategy_returns'])
    all_returns = np.array(all_returns)
    
    # Calculate cumulative equity curve
    equity_curve = np.cumprod(1 + all_returns)
    
    # Total return
    total_return = equity_curve[-1] - 1
    
    # Annualized return (assuming daily returns, 252 trading days/year)
    n_days = len(all_returns)
    n_years = n_days / 252
    annualized_return = (1 + total_return) ** (1 / n_years) - 1
    
    # Volatility (annualized)
    volatility = np.std(all_returns) * np.sqrt(252)
    
    # Sharpe ratio
    excess_return = annualized_return - risk_free_rate
    sharpe_ratio = excess_return / volatility if volatility > 0 else 0
    
    # Maximum drawdown
    running_max = np.maximum.accumulate(equity_curve)
    drawdown = (equity_curve - running_max) / running_max
    max_drawdown = np.min(drawdown)
    
    # Win rate and average win/loss
    winning_trades = all_returns[all_returns > 0]
    losing_trades = all_returns[all_returns < 0]
    
    win_rate = len(winning_trades) / len(all_returns[all_returns != 0]) if len(all_returns[all_returns != 0]) > 0 else 0
    avg_win = np.mean(winning_trades) if len(winning_trades) > 0 else 0
    avg_loss = np.mean(losing_trades) if len(losing_trades) > 0 else 0
    
    # Profit factor
    total_wins = np.sum(winning_trades)
    total_losses = np.abs(np.sum(losing_trades))
    profit_factor = total_wins / total_losses if total_losses > 0 else np.inf
    
    return {
        'total_return': total_return,
        'annualized_return': annualized_return,
        'volatility': volatility,
        'sharpe_ratio': sharpe_ratio,
        'max_drawdown': max_drawdown,
        'win_rate': win_rate,
        'avg_win': avg_win,
        'avg_loss': avg_loss,
        'profit_factor': profit_factor,
        'n_trades': len(all_returns[all_returns != 0]),
        'n_winning': len(winning_trades),
        'n_losing': len(losing_trades)
    }

print("Performance metrics functions ready")

Performance metrics functions ready


In [21]:
# Calculate performance metrics for all strategies
performance_summary = {}

print("="*80)
print("PERFORMANCE SUMMARY")
print("="*80)

for strategy in strategies:
    results = results_by_strategy[strategy]
    metrics = calculate_performance_metrics(results)
    performance_summary[strategy] = metrics
    
    print(f"\n{strategy.upper()}:")
    print("-" * 80)
    print(f"  Total Return:        {metrics['total_return']*100:8.2f}%")
    print(f"  Annualized Return:   {metrics['annualized_return']*100:8.2f}%")
    print(f"  Volatility (annual): {metrics['volatility']*100:8.2f}%")
    print(f"  Sharpe Ratio:        {metrics['sharpe_ratio']:8.3f}")
    print(f"  Max Drawdown:        {metrics['max_drawdown']*100:8.2f}%")
    print(f"  Win Rate:            {metrics['win_rate']*100:8.2f}%")
    print(f"  Average Win:         {metrics['avg_win']*100:8.4f}%")
    print(f"  Average Loss:        {metrics['avg_loss']*100:8.4f}%")
    print(f"  Profit Factor:       {metrics['profit_factor']:8.3f}")
    print(f"  Total Trades:        {metrics['n_trades']:8d}")
    print(f"    Winning:           {metrics['n_winning']:8d}")
    print(f"    Losing:            {metrics['n_losing']:8d}")

print("\n" + "="*80)

# Save performance summary
with open('performance_summary.json', 'w') as f:
    json.dump(performance_summary, f, indent=2)

print("\nPerformance summary saved to: performance_summary.json")

PERFORMANCE SUMMARY

BUY_SELL:
--------------------------------------------------------------------------------
  Total Return:           -7.71%
  Annualized Return:      -6.16%
  Volatility (annual):     7.99%
  Sharpe Ratio:          -0.771
  Max Drawdown:          -12.42%
  Win Rate:               50.00%
  Average Win:           0.5114%
  Average Loss:         -0.6055%
  Profit Factor:          0.845
  Total Trades:             162
    Winning:                 81
    Losing:                  81

ONLY_BUY:
--------------------------------------------------------------------------------
  Total Return:            3.33%
  Annualized Return:       2.63%
  Volatility (annual):     5.69%
  Sharpe Ratio:           0.462
  Max Drawdown:           -5.41%
  Win Rate:               53.09%
  Average Win:           0.5634%
  Average Loss:         -0.5459%
  Profit Factor:          1.168
  Total Trades:              81
    Winning:                 43
    Losing:                  38

ONLY_SELL:
--

In [22]:
import pandas as pd

# Create comparison DataFrame
comparison_data = []
for strategy, metrics in performance_summary.items():
    comparison_data.append({
        'Strategy': strategy.replace('_', ' ').title(),
        'Total Return (%)': f"{metrics['total_return']*100:.2f}",
        'Annual Return (%)': f"{metrics['annualized_return']*100:.2f}",
        'Sharpe Ratio': f"{metrics['sharpe_ratio']:.3f}",
        'Max Drawdown (%)': f"{metrics['max_drawdown']*100:.2f}",
        'Win Rate (%)': f"{metrics['win_rate']*100:.2f}",
        'Trades': metrics['n_trades']
    })

comparison_df = pd.DataFrame(comparison_data)

print("\nStrategy Comparison:")
print("="*100)
print(comparison_df.to_string(index=False))
print("="*100)

# Save comparison table
comparison_df.to_csv('strategy_comparison.csv', index=False)
print("\nComparison table saved to: strategy_comparison.csv")


Strategy Comparison:
 Strategy Total Return (%) Annual Return (%) Sharpe Ratio Max Drawdown (%) Win Rate (%)  Trades
 Buy Sell            -7.71             -6.16       -0.771           -12.42        50.00     162
 Only Buy             3.33              2.63        0.462            -5.41        53.09      81
Only Sell           -10.69             -8.57       -1.532           -11.05        46.91      81

Comparison table saved to: strategy_comparison.csv
